# Classifying Penguins with Keras

In [38]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

In [39]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [40]:
penguins = penguins.dropna().reset_index(drop=True)

In [41]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3750.0,39.1,18.7,181.0,False,True
1,3800.0,39.5,17.4,186.0,True,False
2,3250.0,40.3,18.0,195.0,True,False
3,3450.0,36.7,19.3,193.0,True,False
4,3650.0,39.3,20.6,190.0,False,True
...,...,...,...,...,...,...
328,4000.0,55.8,19.8,207.0,False,True
329,3400.0,43.5,18.1,202.0,True,False
330,3775.0,49.6,18.2,193.0,False,True
331,4100.0,50.8,19.0,210.0,False,True


In [42]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.291667,0.254545,0.666667,0.152542,0.0,1.0
1,0.305556,0.269091,0.511905,0.237288,1.0,0.0
2,0.152778,0.298182,0.583333,0.389831,1.0,0.0
3,0.208333,0.167273,0.738095,0.355932,1.0,0.0
4,0.263889,0.261818,0.892857,0.305085,0.0,1.0
...,...,...,...,...,...,...
328,0.361111,0.861818,0.797619,0.593220,0.0,1.0
329,0.194444,0.414545,0.595238,0.508475,1.0,0.0
330,0.298611,0.636364,0.607143,0.355932,0.0,1.0
331,0.388889,0.680000,0.702381,0.644068,0.0,1.0


In [43]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()

features = torch.tensor(scaled_penguins_x.to_numpy(dtype=np.float32))
targets = torch.tensor(penguins_y, dtype=torch.long)
full_dataset = TensorDataset(features, targets)

val_size = int(len(full_dataset) * 0.1)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False)

def keras_batches(loader):
    while True:
        for x_batch, y_batch in loader:
            yield x_batch.numpy(), y_batch.numpy()

penguins_y

0         Adelie
1         Adelie
2         Adelie
3         Adelie
4         Adelie
         ...    
328    Chinstrap
329    Chinstrap
330    Chinstrap
331    Chinstrap
332    Chinstrap
Name: species, Length: 333, dtype: object


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [44]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [45]:
model.summary()

Model: "penguin_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 6)]               0         
                                                                 
 dense_8 (Dense)             (None, 7)                 49        
                                                                 
 dense_9 (Dense)             (None, 5)                 40        
                                                                 
 dense_10 (Dense)            (None, 3)                 18        
                                                                 
 dense_11 (Dense)            (None, 3)                 12        
                                                                 
Total params: 119 (476.00 Byte)
Trainable params: 119 (476.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [46]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [47]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(
    keras_batches(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=100,
    validation_data=keras_batches(val_loader),
    validation_steps=len(val_loader),
)

scores = model.evaluate(keras_batches(full_loader), steps=len(full_loader), verbose=2)

Epoch 1/100


c:\Users\jfigg\.conda\envs\keras_env\Lib\site-packages\keras\src\backend.py:5714: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 [==============================] - 1s 76ms/step - loss: 1.0938 - accuracy: 0.3933 - val_loss: 1.0916 - val_accuracy: 0.5152
Epoch 2/100
5/5 [==============================] - 0s 16ms/step - loss: 1.0902 - accuracy: 0.4433 - val_loss: 1.0898 - val_accuracy: 0.4848
Epoch 3/100
5/5 [==============================] - 0s 15ms/step - loss: 1.0878 - accuracy: 0.4333 - val_loss: 1.0884 - val_accuracy: 0.4848
Epoch 4/100
5/5 [==============================] - 0s 14ms/step - loss: 1.0855 - accuracy: 0.4333 - val_loss: 1.0869 - val_accuracy: 0.4848
Epoch 5/100
5/5 [==============================] - 0s 18ms/step - loss: 1.0834 - accuracy: 0.4333 - val_loss: 1.0855 - val_accuracy: 0.4848
Epoch 6/100
5/5 [==============================] - 0s 15ms/step - loss: 1.0813 - accuracy: 0.4333 - val_loss: 1.0842 - val_accuracy: 0.4848
Epoch 7/100
5/5 [==============================] - 0s 17ms/step - loss: 1.0792 - accuracy: 0.4333 - val_loss: 1.0827 - val_accuracy: 0.4848
Epoch 8/100
5/5 [===============

In [53]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(
    keras_batches(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=100,
    validation_data=keras_batches(val_loader),
    validation_steps=len(val_loader),
)

scores = model_logit_true.evaluate(keras_batches(full_loader), steps=len(full_loader), verbose = 2)

Epoch 1/100
5/5 [==============================] - 1s 61ms/step - loss: 0.3356 - accuracy: 0.8433 - val_loss: 0.4569 - val_accuracy: 0.6970
Epoch 2/100
5/5 [==============================] - 0s 18ms/step - loss: 0.3333 - accuracy: 0.8533 - val_loss: 0.4565 - val_accuracy: 0.6970
Epoch 3/100
5/5 [==============================] - 0s 18ms/step - loss: 0.3333 - accuracy: 0.8500 - val_loss: 0.4554 - val_accuracy: 0.6970
Epoch 4/100
5/5 [==============================] - 0s 16ms/step - loss: 0.3313 - accuracy: 0.8567 - val_loss: 0.4549 - val_accuracy: 0.6970
Epoch 5/100
5/5 [==============================] - 0s 15ms/step - loss: 0.3305 - accuracy: 0.8533 - val_loss: 0.4554 - val_accuracy: 0.6970
Epoch 6/100
5/5 [==============================] - 0s 16ms/step - loss: 0.3298 - accuracy: 0.8433 - val_loss: 0.4531 - val_accuracy: 0.6970
Epoch 7/100
5/5 [==============================] - 0s 16ms/step - loss: 0.3286 - accuracy: 0.8467 - val_loss: 0.4513 - val_accuracy: 0.6970
Epoch 8/100
5/5 [===

In [49]:
model_logit_true.predict(scaled_penguins_x)

11/11 [==============================] - 0s 2ms/step


array([[8.93261373e-01, 1.06719762e-01, 1.88058966e-05],
       [8.27006996e-01, 1.72605395e-01, 3.87688546e-04],
       [8.25148225e-01, 1.74436331e-01, 4.15366259e-04],
       [9.16729152e-01, 8.32667127e-02, 4.20173501e-06],
       [9.22641218e-01, 7.73560032e-02, 2.70906321e-06],
       [8.73775303e-01, 1.26171872e-01, 5.27817210e-05],
       [8.79898012e-01, 1.20063156e-01, 3.87937325e-05],
       [8.44251335e-01, 1.55550808e-01, 1.97819711e-04],
       [9.31367874e-01, 6.86307251e-02, 1.33369656e-06],
       [9.39473927e-01, 6.05254173e-02, 6.37213589e-07],
       [8.84580731e-01, 1.15388870e-01, 3.03609831e-05],
       [8.85057628e-01, 1.14912756e-01, 2.95978043e-05],
       [8.84138525e-01, 1.15830429e-01, 3.10837495e-05],
       [9.18272436e-01, 8.17238316e-02, 3.75762716e-06],
       [8.89472425e-01, 1.10504240e-01, 2.32755010e-05],
       [9.13330197e-01, 8.66645426e-02, 5.33835964e-06],
       [9.03205633e-01, 9.67840031e-02, 1.03820057e-05],
       [9.23514783e-01, 7.64826

In [50]:
penguins['species']

0         Adelie
1         Adelie
2         Adelie
3         Adelie
4         Adelie
         ...    
328    Chinstrap
329    Chinstrap
330    Chinstrap
331    Chinstrap
332    Chinstrap
Name: species, Length: 333, dtype: object

In [51]:
penguins_y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,